In [3]:
import pandas as pd
import os

In [22]:
os.makedirs('tables', exist_ok=True)

### Pretrained

In [8]:
pretrained_nott = pd.read_csv('results/results_pretrained_nottingham_mae.csv')
pretrained_jazz = pd.read_csv('results/results_pretrained_jazz_mae.csv')
pretrained_jazz.head(20)

,Instance,CHE,CC,CTD,CTnCTR,PCS,MCTD,HRHE_i,HRC_i,CBS,avg
0,pretrained,0.926175,6.321429,0.324133,0.189083,0.107426,0.151133,0.470664,0.964286,0.193271,1.071955
1,pretrained_epoch168_nvis50,0.736943,5.857143,0.294047,0.180832,0.127954,0.176455,0.446134,0.964286,0.200442,0.998248
2,pretrained_epoch177_nvis30,0.739771,5.892857,0.205565,0.190135,0.116383,0.165577,0.385624,1.107143,0.161555,0.996068
3,pretrained_epoch182_nvis20,0.656791,5.035714,0.243619,0.183351,0.116788,0.161136,0.491665,1.142857,0.155669,0.909732
4,pretrained_epoch189_nvis10,0.583134,4.821429,0.212568,0.188974,0.127758,0.182253,0.401069,1.392857,0.131610,0.893517
5,pretrained_epoch196_nvis5,0.590991,4.714286,0.246363,0.173874,0.130795,0.163590,0.482840,1.214286,0.118951,0.870664
6,pretrained_epoch200_nvis3,0.588331,4.964286,0.251322,0.171879,0.120800,0.170016,0.482180,1.357143,0.140919,0.916320
7,pretrained_epoch203_nvis2,0.617748,4.928571,0.262810,0.191627,0.120350,0.183562,0.526791,1.500000,0.161387,0.943650
8,pretrained,0.408245,3.928571,0.155453,0.083010,0.090006,0.071431,0.549074,1.642857,0.155997,0.787183
9,pretrained_back,0.502708,4.178571,0.217160,0.067355,0.096864,0.074468,0.336037,0.714286,0.139116,0.702952


In [20]:
# Combine the two pretrained results on the common `Instance` column.
# Use a duplicate counter so repeated Instance values align row-by-row,
# and preserve the original order from the Nottingham dataframe.
nott = pretrained_nott[['Instance', 'avg']].rename(columns={'avg': 'Nott'})
nott['Instance_order'] = nott.groupby('Instance').cumcount()
nott['merge_order'] = range(len(nott))

jazz = pretrained_jazz[['Instance', 'avg']].rename(columns={'avg': 'Jazz'})
jazz['Instance_order'] = jazz.groupby('Instance').cumcount()

combined_pretrained = (
    nott.merge(
        jazz,
        on=['Instance', 'Instance_order'],
        how='inner',
        sort=False
    )
    .sort_values('merge_order')
    .drop(columns=['Instance_order', 'merge_order'])
)

# Add the overall average of the new Nott and Jazz columns.
combined_pretrained['avg'] = combined_pretrained[['Nott', 'Jazz']].mean(axis=1)

# Create a renamed `new_name` column using the requested formula.
def rename_instance(current_name):
    tmp_split = current_name.split('_')
    if len(tmp_split) > 2:
        tmp_split[1] = tmp_split[1].replace('epoch', 'e')
        tmp_split[2] = tmp_split[2].replace('nvis', 'v')
        return tmp_split[1] + '(' + tmp_split[2] + ')'
    else:
        return current_name

combined_pretrained['Instance'] = combined_pretrained['Instance'].map(rename_instance)

In [21]:
combined_pretrained.head(20)

,Instance,Nott,Jazz,avg
0,pretrained,0.542511,1.071955,0.807233
1,e168(v50),0.533198,0.998248,0.765723
2,e177(v30),0.509708,0.996068,0.752888
3,e182(v20),0.495584,0.909732,0.702658
4,e189(v10),0.532162,0.893517,0.712839
5,e196(v5),0.460414,0.870664,0.665539
6,e200(v3),0.428531,0.916320,0.672425
7,e203(v2),0.485003,0.943650,0.714326
8,pretrained,0.508742,0.787183,0.647962
9,pretrained_back,0.318852,0.702952,0.510902
